# 09 — 多 Agent 協作:主管分派任務給專員(Supervisor Pattern)

**這份要學什麼**
- Supervisor pattern：一個路由節點分派任務給多個專員 node / subgraph
- `Command(goto=...)` 在多 agent 場景怎麼派上用場

> 不需要 API key:負責「決定交給誰」的 supervisor 用 `scripted_model` 模擬一個已經照劇本
> 決定好台詞的 LLM,機制跟接上真模型時完全一樣,只是這裡的決策是照劇本走、不是真的思考。

**情境比喻**:你在公司遇到問題,不會直接去堵會計專員問技術問題。你會先找主管,主管聽完
問題,判斷「這件事該找誰」,轉介給對的專員處理;專員辦完把結果回報給主管,主管看完再決定
下一步——繼續轉給別的專員,還是可以回覆你了。這就是 **supervisor pattern**:一個「路由
節點」負責分派,其他節點只專心做自己那一小塊。

```
                    ┌───────────────┐
                    │  supervisor    │  ← 主管:看對話,決定換誰做
                    │  (決定交給誰)   │
                    └───────┬────────┘
                 ┌──────────┼──────────┐
                 ▼                     ▼
         ┌────────────────┐   ┌────────────────┐
         │   researcher    │   │     writer      │
         │  (查資料的專員)  │   │  (寫文章的專員)  │
         │  = 一個 subgraph │   │  = 一個 subgraph │
         └────────┬─────────┘   └────────┬────────┘
                  │ 辦完回報              │ 辦完回報
                  └───────────┬──────────┘
                              ▼
                        回到 supervisor
```

這份 notebook 把兩個東西合起來用:

- **`Command(goto=...)`**(`04_langgraph_control_flow.ipynb`):supervisor 用它表達
  「我看完現況了,決定接下來換誰上場」
- **Subgraph(子圖)**:把「研究員」「寫手」各自的一整套處理流程,包成一個獨立編譯好的
  `StateGraph`,再整個塞進外層圖裡當「一個 node」用——外層的 supervisor 完全不用管
  子圖內部怎麼運作,只在乎丟進去什麼、吐出來什麼。條件是子圖跟外層圖要共用同一組 State
  欄位(這裡兩邊都用 `messages`),不然資料傳不過去

**這個架構之後會再遇到一次**:14 章的 capstone 案例(billing 專員 + tech 專員 + 一個
supervisor)用的就是一模一樣的骨架,只是專員換成處理帳務/技術問題的兩個角色。

In [1]:
import sys

sys.path.insert(0, ".")
from _llm import scripted_model

from typing import Literal

from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.types import Command

## 兩個專職「部門」,各自包成一個 subgraph

把 researcher、writer 想像成公司裡的兩個部門,各自有自己的 SOP(這裡簡化成只有一個步驟,
現實中可能是一整個 ReAct agent,見 `05`)。重點不是內部多複雜,而是「supervisor 之外還有
一層圖」這個結構——每個部門都是獨立編譯好的 `StateGraph`,被當成外層圖的一個 node 塞進去。

In [2]:
def research_step(state: MessagesState) -> dict:
    return {"messages": [("ai", "[researcher] 查到三筆相關資料")]}


researcher_builder = StateGraph(MessagesState)
researcher_builder.add_node("research_step", research_step)
researcher_builder.add_edge(START, "research_step")
researcher_builder.add_edge("research_step", END)
researcher_subgraph = researcher_builder.compile()


def write_step(state: MessagesState) -> dict:
    return {"messages": [("ai", "[writer] 文章初稿完成")]}


writer_builder = StateGraph(MessagesState)
writer_builder.add_node("write_step", write_step)
writer_builder.add_edge(START, "write_step")
writer_builder.add_edge("write_step", END)
writer_subgraph = writer_builder.compile()

## Supervisor:讀對話、決定交給誰

現實情況下,這裡會是把目前對話丟給一個真的 LLM,請它從 `["researcher", "writer", "end"]`
三個選項裡選一個(通常會用 `response_format` 限制輸出只能是這幾個字之一,比要求 LLM 自己
生文字、再用程式解析穩定得多)。

這裡的 `router_model` 是 `scripted_model`,等於一個「台詞都寫好的演員」,依序回傳
`"researcher" -> "writer" -> "end"`,模擬 supervisor 做了三輪決策的過程——決策內容是照
劇本走,但 `Command(goto=...)` 路由的機制跟接真 LLM 時完全一樣。

In [3]:
router_model = scripted_model(["researcher", "writer", "end"])


def supervisor(state: MessagesState) -> Command[Literal["researcher", "writer", "__end__"]]:
    decision = router_model.invoke(state["messages"]).content.strip()
    if decision == "end":
        return Command(goto=END)
    return Command(goto=decision, update={"messages": [("ai", f"[supervisor] 交給 {decision}")]})

## 組裝:supervisor 在中間,兩個 subgraph 做完都繞回 supervisor 再決定下一步

畫成公司架構就是:supervisor 是主管,researcher / writer 是兩個部門,部門辦完事永遠先
回報給主管,由主管決定「還要繼續轉介,還是可以結束了」——不會有部門辦完事直接回覆使用者。
下面印出的 mermaid 圖裡,虛線是 supervisor 的「決定」(可能去 researcher、writer 或結束),
實線是部門辦完後「一定會走」的回報路徑。

In [ ]:
from _graph_viz import show_graph

top_builder = StateGraph(MessagesState)
top_builder.add_node("supervisor", supervisor)
top_builder.add_node("researcher", researcher_subgraph)
top_builder.add_node("writer", writer_subgraph)
top_builder.add_edge(START, "supervisor")
top_builder.add_edge("researcher", "supervisor")
top_builder.add_edge("writer", "supervisor")
team = top_builder.compile()

show_graph(team)

In [5]:
result = team.invoke({"messages": [("human", "幫我研究這個主題，然後寫一篇文章")]})
for m in result["messages"]:
    print(type(m).__name__, "|", m.content)

HumanMessage | 幫我研究這個主題，然後寫一篇文章
AIMessage | [supervisor] 交給 researcher
AIMessage | [researcher] 查到三筆相關資料
AIMessage | [supervisor] 交給 writer
AIMessage | [writer] 文章初稿完成


追蹤上面的輸出,對照劇本 `["researcher", "writer", "end"]` 的順序,實際跑起來是這樣:

```
使用者:「幫我研究這個主題,然後寫一篇文章」
   │
   ▼
[supervisor] 交給 researcher ──▶ [researcher] 查到三筆相關資料
   │                                          │
   ◀──────────────────回報───────────────────┘
   ▼
[supervisor] 交給 writer ──▶ [writer] 文章初稿完成
   │                                    │
   ◀─────────────────回報─────────────┘
   ▼
[supervisor] 決定 end → 結束,回覆使用者
```

## 為什麼用 subgraph,而不是把所有 node 攤平在同一個圖裡
- **封裝**:`researcher_subgraph` 內部想加幾個步驟、要不要用 `ToolNode`,都是它自己的事,
  外層 supervisor 不用關心,就像你不用知道會計部門內部用哪套軟體做帳
- **State 隔離**:子圖可以有自己額外的內部欄位,只要跟外層共用的欄位(這裡是 `messages`)
  對得上,就能跟外層圖互通資料
- **可重用**:同一個 subgraph 可以被多個外層圖拿去組裝,跟寫程式時抽出共用函式是同一個
  道理——14 章的 capstone 就是把這套「supervisor + 專員 subgraph」的骨架原封不動搬過去,
  只是換成 billing / tech 兩個專員

## 小結
- Supervisor pattern = 一個負責路由的節點(主管)+ 多個專職節點或 subgraph(專員)
- `Command(goto=...)` 讓一個節點同時「做決定」跟「導向下一步」,是表達路由決策最直覺的寫法
- Subgraph 讓多 agent 系統可以像組織部門一樣,一層包一層地組起來

最後一份:`10_langgraph_persistence_deploy.ipynb`,把 checkpointer 換成正式環境能用的
持久化後端,並簡介部署概念。